In [12]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()
import pickle

In [3]:
prompt='''Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.'''

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.6-luna",
    input=prompt
)

In [5]:
response.output[0]

ResponseOutputMessage(id='msg_0d7ac637d4cb4481006aa985a8ed6c87d1b89206abc16efc39', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [6]:
response.output[0].content[0].text

'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [7]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [9]:
response.usage

ResponseUsage(input_tokens=179, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=30, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=209)

In [10]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00026925

In [11]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [13]:

with open("documents.pkl",'rb') as f:
    documents=pickle.load(f)

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
# search_results

In [18]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [19]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [20]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [21]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [22]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [ ]:

def rag(query, model="gpt-5.6-luna"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [24]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [25]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort.\n\nTo qualify, you need to:\n- complete the capstone project\n- finish the required peer reviews\n\nHomework is not required for the certificate.\n\nYou can prepare in self-paced mode, but the project submission and peer reviews must happen while a live cohort is accepting them.'